# Baseline dự đoán lưu lượng giao thông (Theo hướng Time-only)

Mục tiêu: dự đoán traffic_volume (số xe/giờ).

Đặc trưng sử dụng: giờ, thứ, tháng và cuối tuần.
Hai phương pháp:
1. Trung bình lưu lượng theo thứ và giờ, tính từ tập train.
2. Ridge Regression với đặc trưng thời gian.

Dữ liệu được chia theo thời gian: 70% train, 15% validation,
15% test. Tập test được giữ lại để đánh giá cuối.

## Quy trình thực hiện
0. Thiết lập môi trường và đọc dữ liệu
1. Kiểm tra thuộc tính dữ liệu và xử lý dữ liệu
2. Chia dữ liệu theo thời gian
3. Tạo đặc trưng thời gian và xác định X, y
4. Huấn luyện baseline trung bình thứ–giờ và Ridge
5. Đánh giá trên validation, ghi log và nhận xét

In [14]:
from pathlib import Path
import hashlib
import sys
import pandas as pd
import numpy as np

## 0. Load dữ liệu
Data location: data\Metro_Interstate_Traffic_Volume.csv

Bộ dữ liệu có 48204 dòng với 9 thuộc tính

### Kiểm tra thông tin dữ liệu đầu vào
- Giả thuyết kiểm tra: notebook đang đọc đúng dataset của dự án.
- Thay đổi: thêm bước đọc CSV và kiểm tra cấu trúc.
- Môi trường: [Python, pandas và kernel từ output].
- Dataset: [đường dẫn và SHA-256].
- Kết quả kiểm tra: [số dòng, số cột, cột bắt buộc].
- Metrics mô hình: chưa áp dụng.
- Vấn đề còn lại: chưa kiểm tra timestamp, mục tiêu và dòng trùng.

In [15]:
# Tìm thư mục dự án khi chạy từ gốc hoặc notebooks/
current_dir = Path.cwd()

if (current_dir / "data").is_dir():
    project_dir = current_dir
elif (current_dir.parent / "data").is_dir():
    project_dir = current_dir.parent
else:
    raise FileNotFoundError("Không tìm thấy thư mục data.")

data_path = (
    project_dir
    / "data"
    / "Metro_Interstate_Traffic_Volume.csv"
)

df_raw = pd.read_csv(data_path, keep_default_na=False) ## dữ liệu thô, chưa xử lý missing value, outlier, feature engineering

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("Dataset:", data_path.resolve())
print("SHA-256:", hashlib.sha256(data_path.read_bytes()).hexdigest())
print("Shape:", df_raw.shape)

display(df_raw.head())

Python executable: e:\Project\SGU_HocMay_2026-2027\.venv\Scripts\python.exe
Python version: 3.14.7
Pandas version: 3.0.6
Dataset: E:\Project\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\data\Metro_Interstate_Traffic_Volume.csv
SHA-256: 749c90d720360a4215bb15345526073c079ba4cc95e3fa558796d083f85fce9e
Shape: (48204, 9)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [16]:
required_columns = {"date_time", "traffic_volume"}
missing_columns = required_columns - set(df_raw.columns)

if missing_columns: #Kiểm tra xem có thiếu cột bắt buộc không
    raise ValueError(f"Thiếu cột bắt buộc: {sorted(missing_columns)}")

if df_raw.empty:
    raise ValueError("Dataset không có dòng dữ liệu.")

if not df_raw.columns.is_unique:
    raise ValueError("DataFrame có tên cột bị trùng.")

print("Số dòng:", len(df_raw))
print("Số cột:", len(df_raw.columns))

display(
    pd.DataFrame({
        "dtype": df_raw.dtypes.astype(str),
        "null_count": df_raw.isna().sum(),
    })
)

Số dòng: 48204
Số cột: 9


,dtype,null_count
holiday,str,0
temp,float64,0
rain_1h,float64,0
snow_1h,float64,0
clouds_all,int64,0
weather_main,str,0
weather_description,str,0
date_time,str,0
traffic_volume,int64,0


$\rightarrow$ Bộ dữ liệu không có dòng nào mang giá trị null và chuỗi rỗng, nhiệt độ bất hợp lý hoặc timestamp không chuyển được vẫn cần kiểm tra sau.

## 1. Kiểm tra các thuộc tính dữ liệu và xử lý dữ liệu

### 1.1. Tiêu chí và quá trình kiểm tra:
1. Có hai cột bắt buộc là date_time và traffic_volume, đảm bảo có thời gian và nhãn dự đoán	và thực hiện thông báo tên cột thiếu
2. Dataset không rỗng, đảm bảo có dữ liệu để thực nghiệm, nếu không chưa thể đi tiếp.
3. Chuyển đổi `date_time` sang dạng datetime(Y-m-d H-M-S) để kiểm tra phát hiện ngày giờ thiếu hoặc sai định dạng.
4. Kiểm tra `traffic_volume` là số hữu hạn để phát hiện chuỗi không chuyển được thành số, giá trị thiếu hoặc vô hạn.
5. Kiểm tra `traffic_volume ≥ 0` để phát hiện lỗi không thuộc miền giá trị cơ bản của số xe.

In [29]:
# 1. Kiểm tra hai cột bắt buộc cho baseline Time-only
required_columns = ["date_time", "traffic_volume"]

missing_columns = set(required_columns) - set(df_raw.columns)

if missing_columns:
    raise ValueError(f"Thiếu cột bắt buộc: {sorted(missing_columns)}")

if df_raw.empty:
    raise ValueError("Dataset không có dòng dữ liệu.")

# 2. Tạo bản xử lý riêng, giữ nguyên df_raw
data = df_raw[required_columns].copy()

data["date_time"] = pd.to_datetime(
    data["date_time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
)

data["traffic_volume"] = pd.to_numeric(
    data["traffic_volume"],
    errors="coerce",
)

# 3. Xác định các dòng không hợp lệ
invalid_time = data["date_time"].isna()
invalid_target = ~np.isfinite(data["traffic_volume"])
negative_target = data["traffic_volume"] < 0

print("Timestamp không hợp lệ:", invalid_time.sum())
print("Mục tiêu thiếu hoặc không hữu hạn:", invalid_target.sum())
print("Mục tiêu âm:", negative_target.sum())

invalid_rows = invalid_time | invalid_target | negative_target

if invalid_rows.any():
    display(df_raw.loc[invalid_rows, required_columns].head(10))
    raise ValueError("Có dữ liệu không hợp lệ; cần kiểm tra trước khi tiếp tục.")

print("Đạt kiểm tra kiểu dữ liệu và miền giá trị cơ bản.")
display(data.head())
print(data.dtypes)

Timestamp không hợp lệ: 0
Mục tiêu thiếu hoặc không hữu hạn: 0
Mục tiêu âm: 0
Đạt kiểm tra kiểu dữ liệu và miền giá trị cơ bản.


,date_time,traffic_volume
0,2012-10-02 09:00:00,5545
1,2012-10-02 10:00:00,4516
2,2012-10-02 11:00:00,4767
3,2012-10-02 12:00:00,5026
4,2012-10-02 13:00:00,4918


date_time         datetime64[us]
traffic_volume             int64
dtype: object


### 1.2. Xử lý dữ liệu gộp timestamp trùng thời gian

Đầu vào: `data` đã vượt qua kiểm tra ở cell trước. Đầu ra: `hourly`, một dòng cho mỗi timestamp, sắp xếp theo thời gian.

Kiểm tra số giá trị `traffic_volume` khác nhau trong từng timestamp trước khi gộp. Chỉ khi mỗi nhóm có đúng một giá trị mục tiêu, giữ giá trị đầu tiên của nhóm. Không cộng vì các dòng cùng timestamp không phải các lượng xe độc lập; không lấy trung bình để che giấu mục tiêu mâu thuẫn.

`row_count`: số dòng gốc trong nhóm. `target_count`: số giá trị mục tiêu khác nhau. `traffic_volume`: giá trị được giữ sau gộp. Bảng cuối hiển thị tối đa 5 nhóm timestamp lặp.

Quy tắc này dành cho baseline Time-only với hai cột đang chọn. Khi bổ sung thời tiết, cần xác định cách tổng hợp riêng cho các cột đó. Đây là gộp timestamp trùng, chưa tạo lại những giờ không có bản ghi.


#### Ví dụ dữ liệu mâu thuẫn giả định

In [30]:
# Ví dụ giả lập: kiểm tra mục tiêu mâu thuẫn tại cùng timestamp
import pandas as pd

example = pd.DataFrame({
    "date_time": [
        "2012-10-10 07:00:00",
        "2012-10-10 07:00:00",
        "2012-10-10 08:00:00",
        "2012-10-10 08:00:00",
        "2012-10-10 09:00:00"
    ],
    "traffic_volume": [
        6793,
        6500,  # Mâu thuẫn với 6793
        6283,
        6283,  # Trùng nhưng không mâu thuẫn
        5680
    ]
})

# Đếm số giá trị mục tiêu khác nhau tại mỗi timestamp
example["target_count"] = (
    example.groupby("date_time")["traffic_volume"]
    .transform("nunique")
)

# Đánh dấu trạng thái
example["status"] = example["target_count"].apply(
    lambda n: "Mâu thuẫn" if n > 1 else "Không mâu thuẫn"
)

display(example)

,date_time,traffic_volume,target_count,status
0,2012-10-10 07:00:00,6793,2,Mâu thuẫn
1,2012-10-10 07:00:00,6500,2,Mâu thuẫn
2,2012-10-10 08:00:00,6283,1,Không mâu thuẫn
3,2012-10-10 08:00:00,6283,1,Không mâu thuẫn
4,2012-10-10 09:00:00,5680,1,Không mâu thuẫn


#### Thực hiện gộp trùng

In [31]:
# Kiểm tra mục tiêu trong từng timestamp trước khi gộp
assert not invalid_rows.any(), "Cần xử lý các dòng không hợp lệ trước khi gộp."

timestamp_summary = data.groupby("date_time").agg(
    row_count=("traffic_volume", "size"),
    target_count=("traffic_volume", "nunique"),
)
conflicting_times = timestamp_summary["target_count"] > 1

print("Timestamp duy nhất:", len(timestamp_summary))
print("Timestamp có nhiều dòng:", (timestamp_summary["row_count"] > 1).sum())
print("Timestamp có mục tiêu mâu thuẫn:", conflicting_times.sum())

if conflicting_times.any():
    # Lấy các timestamp có mục tiêu mâu thuẫn
    conflict_timestamps = timestamp_summary.index[conflicting_times]

    # Lấy các dòng dữ liệu gốc thuộc timestamp đó
    conflict_rows = data[
        data["date_time"].isin(conflict_timestamps)
    ].sort_values("date_time")

    print("Các timestamp có mục tiêu mâu thuẫn:")
    display(timestamp_summary.loc[conflicting_times].head(10))

    print("Chi tiết các dòng mâu thuẫn:")
    display(
        conflict_rows[
            ["date_time", "traffic_volume"]
        ].head(20)
    )

    raise ValueError(
        "Cần kiểm tra mục tiêu mâu thuẫn trước khi gộp."
    )

# Các mục tiêu trong mỗi nhóm đã giống nhau: giữ một giá trị, không cộng
hourly = (
    data.groupby("date_time", as_index=False)
    .agg(traffic_volume=("traffic_volume", "first"))
    .sort_values("date_time")
    .reset_index(drop=True)
)

print("Dòng trước gộp:", len(data))
print("Dòng sau gộp:", len(hourly))
print("Dòng dư đã loại:", len(data) - len(hourly))
print("Timestamp trùng sau gộp:", hourly["date_time"].duplicated().sum())

# Ví dụ nhóm có nhiều dòng và giá trị mục tiêu giữ lại
sample = timestamp_summary[timestamp_summary["row_count"] > 1].head(5)
display(sample.join(hourly.set_index("date_time")))


Timestamp duy nhất: 40575
Timestamp có nhiều dòng: 5445
Timestamp có mục tiêu mâu thuẫn: 0
Dòng trước gộp: 48204
Dòng sau gộp: 40575
Dòng dư đã loại: 7629
Timestamp trùng sau gộp: 0


,row_count,target_count,traffic_volume
date_time,,,
2012-10-10 07:00:00,2,1,6793
2012-10-10 08:00:00,2,1,6283
2012-10-10 09:00:00,2,1,5680
2012-10-14 09:00:00,3,1,2685
2012-10-14 10:00:00,2,1,3370


## 2. Chia dữ liệu theo thời gian

Đầu vào: `hourly`, mỗi timestamp có một dòng và đã được sắp xếp tăng dần. Đầu ra: `train`, `validation`, `test` và bảng `split_summary`.

Với N dòng, hai vị trí cắt là `int(0.70 * N)` và `int(0.85 * N)`. Train lấy 70% đầu; validation lấy 15% tiếp; test lấy phần còn lại. Do làm tròn xuống ở vị trí cắt, tỷ lệ thực tế có thể lệch nhỏ so với 70/15/15. Đây là tỷ lệ theo số timestamp quan sát được, không phải theo độ dài thời gian trên lịch.

`iloc` cắt theo vị trí dòng; điểm cuối lát cắt không được lấy. `.copy()` cho phép tạo thêm cột ở từng tập sau này. Các kiểm tra `assert` yêu cầu timestamp không trùng, đúng thứ tự, ba tập không rỗng, không giao nhau về thời gian và không mất dòng.

Train dùng để tính thống kê baseline, fit tiền xử lý và huấn luyện. Validation dùng để đánh giá trong quá trình phát triển. Test giữ cho đánh giá cuối khi đã chốt phương pháp. Không xáo trộn dữ liệu và không chọn lại mốc chia theo điểm số mô hình.

Nhật ký bước này: lưu chiến lược Chronological, số mẫu, tỷ lệ và mốc đầu/cuối trong `split_summary`; metrics chưa áp dụng. Chưa fit encoding/scaling hoặc huấn luyện mô hình.


### Xét tính liên tục thời gian của bộ dữ liệu, kiểm tra đứt gãy chuỗi thời gian

In [20]:
times = hourly["date_time"]

assert times.notna().all(), "Timestamp có giá trị thiếu."
assert times.is_unique, "Timestamp vẫn còn trùng."
assert times.is_monotonic_increasing, "Thời gian chưa được sắp xếp."
assert times.eq(times.dt.floor("h")).all(), "Timestamp không nằm đúng mốc giờ."

# Khoảng cách giữa hai timestamp liên tiếp
time_diff = times.diff()
gap_mask = time_diff > pd.Timedelta(hours=1)

# Các khoảng bị thiếu giờ
gaps = pd.DataFrame({
    "Trước khoảng trống": times.shift(1)[gap_mask],
    "Sau khoảng trống": times[gap_mask],
})

gaps["Số giờ thiếu"] = (
    (
        gaps["Sau khoảng trống"]
        - gaps["Trước khoảng trống"]
    ) / pd.Timedelta(hours=1) - 1
).astype(int)

print("Timestamp quan sát được:", len(times))
print("Số khoảng đứt gãy:", len(gaps))
print("Tổng số giờ thiếu:", gaps["Số giờ thiếu"].sum())

display(
    gaps.sort_values("Số giờ thiếu", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

Timestamp quan sát được: 40575
Số khoảng đứt gãy: 2588
Tổng số giờ thiếu: 11976


,Trước khoảng trống,Sau khoảng trống,Số giờ thiếu
0,2014-08-08 01:00:00,2015-06-11 20:00:00,7386
1,2013-10-27 01:00:00,2013-11-06 04:00:00,242
2,2015-06-14 20:00:00,2015-06-19 18:00:00,117
3,2014-04-29 08:00:00,2014-05-04 05:00:00,116
4,2015-10-23 11:00:00,2015-10-27 08:00:00,92
5,2014-07-27 16:00:00,2014-07-30 09:00:00,64
6,2013-08-31 00:00:00,2013-09-01 23:00:00,46
7,2015-06-22 18:00:00,2015-06-24 04:00:00,33
8,2014-08-02 02:00:00,2014-08-03 11:00:00,32
9,2013-09-03 02:00:00,2013-09-04 08:00:00,29


### Thực hiện chia train/validation/test

In [21]:

# Chia theo vị trí trên dữ liệu đã sắp xếp thời gian
n_rows = len(hourly)
train_end = int(n_rows * 0.70)
validation_end = int(n_rows * 0.85)
assert 0 < train_end < validation_end < n_rows, "Không đủ dữ liệu để chia ba tập."

train = hourly.iloc[:train_end].copy()
validation = hourly.iloc[train_end:validation_end].copy()
test = hourly.iloc[validation_end:].copy()

split_summary = pd.DataFrame([
    {
        "Tập": name,
        "Số dòng": len(subset),
        "Tỷ lệ (%)": round(100 * len(subset) / n_rows, 2),
        "Bắt đầu": subset["date_time"].min(),
        "Kết thúc": subset["date_time"].max(),
    }
    for name, subset in [
        ("Train", train),
        ("Validation", validation),
        ("Test", test),
    ]
])
display(split_summary)


,Tập,Số dòng,Tỷ lệ (%),Bắt đầu,Kết thúc
0,Train,28402,70.0,2012-10-02 09:00:00,2017-05-10 04:00:00
1,Validation,6086,15.0,2017-05-10 05:00:00,2018-01-19 14:00:00
2,Test,6087,15.0,2018-01-19 15:00:00,2018-09-30 23:00:00


### Tách đặc trưng date_time thành các thành phần đặc trưng nhỏ hơn

Tách thành các đặc trưng dựa theo date_time như weekday(thứ trong tuần), hour(giờ) để đánh giá độ ảnh hưởng của giờ và thứ trong tuần. Thông thường thì lưu lương biến động theo giờ, thứ như giờ cao điểm hay cuối tuần. 

In [22]:
# Tạo bản sao để giữ nguyên các tập sau khi chia
train_time = train.copy()
validation_time = validation.copy()

# Tách thứ trong tuần và giờ từ timestamp
train_time["weekday"] = train_time["date_time"].dt.dayofweek
train_time["hour"] = train_time["date_time"].dt.hour

validation_time["weekday"] = validation_time["date_time"].dt.dayofweek
validation_time["hour"] = validation_time["date_time"].dt.hour

display(
    train_time[["date_time", "weekday", "hour", "traffic_volume"]].head()
)

,date_time,weekday,hour,traffic_volume
0,2012-10-02 09:00:00,1,9,5545
1,2012-10-02 10:00:00,1,10,4516
2,2012-10-02 11:00:00,1,11,4767
3,2012-10-02 12:00:00,1,12,5026
4,2012-10-02 13:00:00,1,13,4918


### Kiểm tra số mẫu theo từng cặp thứ-giờ

In [ ]:
weekday_hour_stats = (
    train_time
    .groupby(["weekday", "hour"])["traffic_volume"]
    .agg(count="count", mean="mean")
)

print("Số nhóm thứ–giờ có dữ liệu:", len(weekday_hour_stats))
print("Số mẫu ít nhất trong một nhóm:", weekday_hour_stats["count"].min())
print("Số mẫu nhiều nhất trong một nhóm:", weekday_hour_stats["count"].max())

display(weekday_hour_stats.head(24)) ## ngày thứ 2

Số nhóm thứ–giờ có dữ liệu: 168
Số mẫu ít nhất trong một nhóm: 155
Số mẫu nhiều nhất trong một nhóm: 183


count         mean
weekday hour                    
0       0       177   617.389831
        1       168   376.029762
        2       174   287.919540
        3       168   335.505952
        4       170   792.541176
        5       175  2549.611429
        6       171  5168.578947
        7       173  5869.069364
        8       165  5252.466667
        9       164  4569.609756
        10      169  4110.236686
        11      167  4324.071856
        12      169  4569.201183
        13      167  4609.323353
        14      166  4925.885542
        15      168  5339.821429
        16      166  5950.156627
        17      166  5623.891566
        18      168  4136.541667
        19      173  3012.254335
        20      168  2552.541667
        21      177  2254.435028
        22      171  1673.216374
        23      173  1098.294798

### Xem nhóm có it mẫu nhất

In [28]:
display(
    weekday_hour_stats
    .sort_values("count")
    .head(10)
)

count         mean
weekday hour                    
2       9       155  5029.929032
1       7       156  6150.032051
2       17      159  6048.729560
        7       160  6173.375000
        13      160  4879.687500
1       5       160  2725.468750
5       6       160  1297.043750
1       9       161  4986.093168
        17      161  5975.981366
3       9       161  5021.012422